In [ ]:
import sys
print("\n".join(sys.path))



In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text, URL
import ollama
import os
from dotenv import load_dotenv

### Get SQL query from LLM and query database, return data as dataframe table

In [ ]:
# 1. Database Connection
url_object = URL.create(
    "postgresql",
    username=os.getenv('DB_USERNAME'),
    password=os.getenv('DB_PASSWORD'),  # SQLAlchemy handles the @ character for you here
    host=os.getenv('DB_HOST'),
    port=int(os.getenv('DB_PORT')),
    database=os.getenv('DB_NAME')
)

engine = create_engine(url_object)

def get_sql_from_llm(user_prompt):
    # 2. Get Schema
    schema_info = pd.read_sql(f"""
        SELECT column_name, data_type 
        FROM information_schema.columns 
        WHERE table_name = 'nyc_pluto_data'
    """, engine)
    schema_str = schema_info.to_string(index=False)
    
    # 3. Construct the System Prompt
    system_message = f"""
    You are a SQL expert. Given the following PostgreSQL table schema, write a SQL query that answers the user's request. 
    Return ONLY the raw SQL code text. Do not include markdown code block formatting (like ```sql) or explanations.
    
    Table: nyc_pluto_data
    Schema: 
    {schema_str}
    """
    
    # 4. Call Ollama
    response = ollama.chat(model='deepseek-r1:14b', messages=[
        {'role': 'system', 'content': system_message},
        {'role': 'user', 'content': user_prompt},
    ])
    
    sql_raw = response['message']['content'].strip()
    
    # --- Clean up Markdown Fences ---
    # Remove starting block
    if sql_raw.startswith("```sql"):
        sql_raw = sql_raw[6:]
    elif sql_raw.startswith("```"):
        sql_raw = sql_raw[3:]
        
    # Remove ending block
    if sql_raw.endswith("```"):
        sql_raw = sql_raw[:-3]
        
    return sql_raw.strip()


# 5. Execute and View Results
query_request = "What are the lowest 5 schooldist values by average assesstot in the nyc_pluto_data table?"
generated_sql = get_sql_from_llm(query_request)

print(f"--- Generated SQL ---\n{generated_sql}\n")

# Run the generated SQL against the database
with engine.connect() as conn:
    result_df = pd.read_sql(text(generated_sql), conn)

print("--- Query Results ---")
print(result_df)
